In [37]:
import json
from pathlib import Path

INPUT_PATH = Path("../../infra/json/kg_extraction/bellicum_contract_kg_normalized.json")
OUTPUT_DIR = Path("../../infra/json/kg_extraction")

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    normalized_kg = json.load(f)

In [38]:
import re
from itertools import combinations

def get_clause_text(ent):
    ev = ent.get("evidence_text", "")
    if isinstance(ev, list):
        return " ".join(ev)
    return str(ev)

def get_clauses(kg):
    return [
        e for e in kg["knowledge_graph"]["entities"]
        if e.get("type") == "Clause"
    ]

def normalize_sentence(s):
    return re.sub(r"\s+", " ", s.lower()).strip()

def split_sentences(text):
    text = re.sub(r"([a-z])\.([A-Z])", r"\1. \2", text)
    return [s.strip() for s in re.split(r"(?<=[.;])\s+", text) if len(s.strip()) > 20]

In [39]:
def detect_condition_polarity_conflict(clause):
    text = get_clause_text(clause)
    sents = split_sentences(text)
    findings = []

    for s1, s2 in combinations(sents, 2):
        a = normalize_sentence(s1)
        b = normalize_sentence(s2)

        pair_text = a + " " + b

        has_only_if = "only if" in pair_text or "so long as" in pair_text
        has_even_if_not = "even if" in pair_text and "not in compliance" in pair_text

        same_topic = (
            "may offer and permit" in pair_text
            or "may permit" in pair_text
            or "may use" in pair_text
            or "right to use" in pair_text
        )

        if has_only_if and has_even_if_not and same_topic:
            findings.append({
                "candidate_type": "CONDITION_POLARITY_CONFLICT",
                "clause_id": clause["id"],
                "clause_label": clause.get("label"),
                "evidence": [s1, s2],
                "reason": "The same permission is conditioned on compliance, but also allowed even when not in compliance."
            })

    return findings

In [40]:
def detect_refundability_conflict(clause):
    text = normalize_sentence(get_clause_text(clause))

    if "non-refundable" in text and "refundable" in text:
        return [{
            "candidate_type": "REFUNDABILITY_CONFLICT",
            "clause_id": clause["id"],
            "clause_label": clause.get("label"),
            "evidence": get_clause_text(clause),
            "reason": "The clause states that an amount is non-refundable and also refundable."
        }]

    return []

In [41]:
def detect_modal_polarity_conflict(clause):
    text = get_clause_text(clause)
    sents = split_sentences(text)
    findings = []

    positive_modals = ["shall ", "must ", "will ", "may "]
    negative_modals = ["shall not ", "must not ", "may not ", "will not ", "not "]

    for s1, s2 in combinations(sents, 2):
        a = normalize_sentence(s1)
        b = normalize_sentence(s2)

        has_positive = any(m in a for m in positive_modals) or any(m in b for m in positive_modals)
        has_negative = any(m in a for m in negative_modals) or any(m in b for m in negative_modals)

        shared_terms = set(a.split()) & set(b.split())
        legal_terms = {
            "use", "supply", "deliver", "purchase", "sell", "terminate",
            "assign", "transfer", "disclose", "pay", "refund", "comply"
        }

        if has_positive and has_negative and len(shared_terms & legal_terms) > 0:
            findings.append({
                "candidate_type": "MODAL_POLARITY_CONFLICT",
                "clause_id": clause["id"],
                "clause_label": clause.get("label"),
                "evidence": [s1, s2],
                "reason": "The clause appears to both allow/require and prohibit a similar action."
            })

    return findings

In [42]:
def extract_numeric_values(text):
    patterns = [
        r"\$[\d,]+(?:\.\d+)?",
        r"€[\d,]+(?:\.\d+)?",
        r"\b\d+\s*(?:days?|months?|years?)\b",
        r"\b\d+\s*%\b",
        r"\bten\s*\(\s*10\s*\)\s*years?\b",
        r"\bfive\s*\(\s*5\s*\)\s*years?\b",
        r"\bsixty\s*\(\s*60\s*\)\s*days?\b",
        r"\bthirty\s*\(\s*30\s*\)\s*days?\b",
    ]

    values = []
    for p in patterns:
        values.extend(re.findall(p, text, flags=re.I))

    return values

def detect_value_conflict(clause):
    text = get_clause_text(clause)
    values = extract_numeric_values(text)

    # Si hay varios valores, no siempre es contradicción.
    # Esto solo genera candidato.
    if len(set(values)) >= 2:
        lower = normalize_sentence(text)

        conflict_markers = [
            "notwithstanding the foregoing",
            "shall not exceed",
            "instead",
            "provided however",
            "unless",
            "except",
        ]

        if any(m in lower for m in conflict_markers):
            return [{
                "candidate_type": "VALUE_CONFLICT_CANDIDATE",
                "clause_id": clause["id"],
                "clause_label": clause.get("label"),
                "values": list(set(values)),
                "evidence": text,
                "reason": "The clause contains multiple values with override/conflict markers."
            }]

    return []

In [43]:
DETECTORS = [
    detect_condition_polarity_conflict,
    detect_refundability_conflict,
    detect_modal_polarity_conflict,
    detect_value_conflict,
]

def detect_intra_clause_contradictions(kg):
    candidates = []

    for clause in get_clauses(kg):
        for detector in DETECTORS:
            candidates.extend(detector(clause))

    return candidates

In [46]:
candidates = detect_intra_clause_contradictions(normalized_kg)

def short_text(x, max_chars=1200):
    if isinstance(x, list):
        x = "\n".join(f"- {s}" for s in x)
    else:
        x = str(x)

    return x[:max_chars] + ("..." if len(x) > max_chars else "")


def print_candidates(candidates):
    print(f"Candidates found: {len(candidates)}")

    for idx, c in enumerate(candidates, start=1):
        print("\n" + "=" * 100)
        print(f"CANDIDATE {idx}/{len(candidates)}")
        print("=" * 100)
        print("Type:", c.get("candidate_type"))
        print("Clause ID:", c.get("clause_id"))
        print("Clause label:", c.get("clause_label"))
        print("Reason:", c.get("reason"))

        if "values" in c:
            print("Values:", c.get("values"))

        print("\nEvidence:")
        print(short_text(c.get("evidence"), max_chars=3000))


##print txt
with open(OUTPUT_DIR / "contradiction_candidates.txt", "w", encoding="utf-8") as f:
    for idx, c in enumerate(candidates, start=1):
        f.write("\n" + "=" * 100 + "\n")
        f.write(f"CANDIDATE {idx}/{len(candidates)}\n")
        f.write("=" * 100 + "\n")
        f.write(f"Type: {c.get('candidate_type')}\n")
        f.write(f"Clause ID: {c.get('clause_id')}\n")
        f.write(f"Clause label: {c.get('clause_label')}\n")
        f.write(f"Reason: {c.get('reason')}\n")

        if "values" in c:
            f.write(f"Values: {c.get('values')}\n")

        f.write("\nEvidence:\n")
        f.write(short_text(c.get("evidence"), max_chars=3000) + "\n")

In [45]:
OUTPUT_CANDIDATES_PATH = OUTPUT_DIR / "bellicum_contradiction_candidates_raw.json"

with open(OUTPUT_CANDIDATES_PATH, "w", encoding="utf-8") as f:
    json.dump(candidates, f, indent=2, ensure_ascii=False)

print("Saved:", OUTPUT_CANDIDATES_PATH)

Saved: ../../infra/json/kg_extraction/bellicum_contradiction_candidates_raw.json
